# Study 3 (clutter-vlm) on Kaggle: Qwen3-VL-8B and Gemma 4 E4B

Runs the experiment of [zoyakhan51/clutter-vlm](https://github.com/zoyakhan51/clutter-vlm) unchanged with one model per notebook, set by `MODEL`: `qwen3vl8b` (`Qwen/Qwen3-VL-8B-Instruct`) or `gemma4e4b` (`google/gemma-4-E4B-it`), both with their weights revision pinned in `models.py`. Each model runs in its own Kaggle notebook. The code is cloned from [ztalat12/clutter-vlm](https://github.com/ztalat12/clutter-vlm) at a fixed commit, and the notebook checks that the scripts are byte-identical to Zoya's commit `cadcd07` before anything runs.

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **on**, phone-verified Kaggle account, secret **`HF_TOKEN`** (Add-ons → Secrets, then switch it on for this notebook): a read token of a Hugging Face account that can see the `atml-shapes-vqa` datasets. Before loading the model, the notebook checks that it can read all four datasets, so a missing token costs seconds, not GPU hours.

**Steps**
1. **Trial:** keep `STAGE = "trial"`, then Save Version → Save & Run All (about 20-40 min). At the end the notebook checks the answers and **estimates how long the full run takes** and whether it fits in Kaggle's 12-hour limit.
2. **Main run:** set `STAGE = "full"` and the `PART` of the run, following Zoya's runner's guide (each part is its own version):
   - Qwen3-VL-8B: `PART = "1"` (answers for `shapes` and `hard`), then `PART = "2"` (answers for `objects` and `lines`, then the naming step).
   - Gemma 4 E4B: `PART = "answers"` (answers for all four datasets, about 8 h), then `PART = "naming"` (the naming step, about 3 h).
   - `PART = "all"` runs everything in one version; only for a model that fits in Kaggle's 12-hour limit.
   If the output of the model's earlier part is attached (Add Input → Your Work → this notebook), its files are copied in and the analysis runs over everything. Otherwise the parts' `preds` files are put together afterwards.
3. **Results:** `/kaggle/working/preds` (answers, scores, nameability, `environment_<MODEL>.json`), `/kaggle/working/results` (`results_<MODEL>.json`, `summary_<MODEL>.txt`) and the full log `log_<MODEL>_*.txt`. Download them or share the notebook with the study owner.

In [ ]:
# ---- settings: the only cell to edit ----
STAGE = "trial"   # "trial" (first 20 images of each version) or "full"
PART = "all"      # "all"; Qwen3-VL-8B: "1" then "2"; Gemma 4 E4B: "answers" then "naming"
MODEL = "qwen3vl8b"  # "qwen3vl8b" (Qwen3-VL-8B) or "gemma4e4b" (Gemma 4 E4B), one notebook per model
REPO = "https://github.com/ztalat12/clutter-vlm"
COMMIT = "a2a127a3875ba6fc57a504a029e79b455a57792d"  # holds Zoya's code of cadcd07, byte for byte
assert STAGE in ("trial", "full") and PART in ("all", "1", "2", "answers", "naming")
assert MODEL in ("qwen3vl8b", "gemma4e4b"), "this study run is for qwen3vl8b or gemma4e4b"
assert not (STAGE == "trial" and PART != "all"), "run the trial with PART = 'all'"
print(f"stage {STAGE}, part {PART}, model {MODEL}, commit {COMMIT[:8]}")

In [ ]:
# both need internet: if pypi.org or github.com cannot be reached, switch Internet on (editor, right panel, Settings)
!pip install -q transformers==5.22.1
!rm -rf /tmp/code && git clone -q {REPO} /tmp/code && cd /tmp/code && git checkout -q {COMMIT} && git log -1 --oneline
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# the scripts must be byte-identical to zoyakhan51/clutter-vlm at cadcd07, so this is exactly the same experiment
import hashlib
from pathlib import Path

ZOYA_CADCD07 = {
    "models.py": "2d281e5694938752503c7f67900d9777a40ac32df8f449bdafb732dcff291765",
    "run_predictions.py": "2db360f79d103b6981fcc178333f6771438824aabc914ff603307cb655e151c8",
    "run_nameability.py": "f3cb335cc4cd3ae4578dd658614450675c9c3b310e5918a406fffe379a4071f2",
    "analysis.py": "72b563c2d0d0a7ef3c85f59eaf12941254c4797ce530d4c6a62c0fbecde52d18",
}
if not (Path("/tmp/code") / "models.py").exists():
    raise RuntimeError("the code could not be downloaded from GitHub (see the cell above): this run has no internet. "
                       "In the editor's right panel, Settings, switch Internet on, then Save Version -> Save & Run All")
import transformers
assert transformers.__version__ == "5.22.1", f"transformers {transformers.__version__} instead of 5.22.1: the pip install above failed"
for file, expected in ZOYA_CADCD07.items():
    found = hashlib.sha256((Path("/tmp/code") / file).read_bytes()).hexdigest()
    assert found == expected, f"{file} differs from Zoya's cadcd07: not the same experiment, stopping"
print("all four scripts are byte-identical to zoyakhan51/clutter-vlm cadcd07")

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()  # a pasted space or newline breaks it
    print("HF_TOKEN secret found")
except Exception as error:
    # without the secret, a token file in an attached private dataset is used (never printed)
    token_files = sorted(Path("/kaggle/input").rglob("hf_token.txt"))
    if token_files:
        os.environ["HF_TOKEN"] = token_files[0].read_text().strip()
        print("HF_TOKEN read from the attached private dataset")
    else:
        print(f"no HF_TOKEN secret attached to this notebook ({type(error).__name__}), trying without one")


def token_problem():
    """Say which account the token belongs to and why it may not see the datasets, without showing the token"""
    from huggingface_hub import whoami
    try:
        me = whoami()
    except Exception as error:
        return (f"Hugging Face does not accept this token at all ({type(error).__name__}): the HF_TOKEN value is wrong "
                "or the token was deleted. Copy it again from huggingface.co -> Settings -> Access Tokens")
    orgs = [org.get("name") for org in me.get("orgs", [])]
    role = me.get("auth", {}).get("accessToken", {}).get("role", "unknown")
    lines = [f"the token belongs to Hugging Face account '{me.get('name')}', token type '{role}', organisations {orgs}"]
    if "atml-shapes-vqa" not in orgs:
        lines.append("-> this account is not a member of atml-shapes-vqa: accept the invite or ask Zoya to add it")
    if role == "fineGrained":
        lines.append("-> fine-grained token: make a token of type Read instead (Settings -> Access Tokens -> Create new token -> Read) "
                     "and put it in the HF_TOKEN secret")
    return "\n".join(lines)


# every dataset is checked before the model is loaded, so a missing token costs seconds, not gpu hours
for dataset in ("shapes-vqa-distractors", "shapes-vqa-hard", "objects-vqa", "lines-vqa"):
    try:
        # the small scorer file of each dataset, which run_predictions.py downloads anyway
        hf_hub_download(f"atml-shapes-vqa/{dataset}", "vqa_scoring.py", repo_type="dataset", local_dir="/tmp/access_check")
    except Exception as error:
        print(f"cannot read atml-shapes-vqa/{dataset}: {type(error).__name__}: {(str(error).splitlines() or [''])[0]}")
        if "HF_TOKEN" in os.environ:
            print(token_problem())
        raise RuntimeError(f"cannot read atml-shapes-vqa/{dataset}, see the reason above. Without a token: add the HF_TOKEN "
                           "secret (Add-ons -> Secrets), tick it for this notebook and start with Save & Run All") from None
print("all four datasets readable")
os.environ["PYTHONUNBUFFERED"] = "1"            # the scripts' prints show up as they happen
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # keeps the log readable
LIMIT = ["--per-version", "20"] if STAGE == "trial" else []
PREDS = Path("/kaggle/working/preds")
PREDS.mkdir(parents=True, exist_ok=True)
LOG = Path(f"/kaggle/working/log_{MODEL}_{STAGE}_part{PART}.txt")
OUTPUT = []


def run(script, *args, must_pass=True):
    """Run one of the study scripts, show its output live, keep it in the log and return its exit code"""
    command = [sys.executable, script, *args]
    print("$ " + " ".join(command), flush=True)
    with open(LOG, "a", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd="/tmp/code", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            OUTPUT.append(line)
        code = process.wait()
    if code and must_pass:
        raise RuntimeError(f"{script} stopped with exit code {code}, see the output above")
    return code


# what each part runs: the datasets it answers and whether it runs the naming step (zoya's runner's guide)
PARTS = {
    "all": (["shapes", "hard", "objects", "lines"], True),
    "1": (["shapes", "hard"], False),                          # qwen3vl8b part A
    "2": (["objects", "lines"], True),                         # qwen3vl8b part B
    "answers": (["shapes", "hard", "objects", "lines"], False),  # gemma4e4b part 1
    "naming": ([], True),                                      # gemma4e4b part 2
}
ANSWERS, NAMING = PARTS[PART]
# a part can run on its own or next to the model's other part. if the other part's output is attached, its files
# are copied in so the analysis can run over everything; otherwise the parts' preds files are put together later
earlier = []
for name in ["shapes", "hard", "objects", "lines"]:
    if name in ANSWERS:
        continue
    for file in (f"preds_{name}_{MODEL}.csv", f"scores_{name}_{MODEL}.csv"):
        found = sorted(Path("/kaggle/input").rglob(file))
        if found:
            shutil.copy(found[0], PREDS / file)
            earlier.append(found[0])
if not NAMING:
    found = sorted(Path("/kaggle/input").rglob(f"nameability_{MODEL}.csv"))
    if found:
        shutil.copy(found[0], PREDS / found[0].name)
        earlier.append(found[0])
for file in earlier:
    print("copied from the attached earlier part:", file)
for environment in sorted({f.parent / f"environment_{MODEL}.json" for f in earlier}):
    if environment.exists():
        other = json.loads(environment.read_text())
        assert other["per_version"] == 0, "the attached output is a trial run, attach the full run of the earlier part"
        shutil.copy(environment, PREDS / f"environment_{MODEL}_earlier_part.json")
print(f"part {PART}: answers for {ANSWERS or 'no dataset'}, naming step {'yes' if NAMING else 'no'}")
print("ready, log at", LOG)

In [ ]:
# answers on the datasets of this part, each scored with the dataset's official vqa_scoring.py
if ANSWERS:
    run("run_predictions.py", "--model", MODEL, "--split", "all", "--datasets", ",".join(ANSWERS), *LIMIT)
else:
    print(f"part {PART}: no answers in this part")

In [ ]:
# what the model calls every squiggle and other_shapes shape of shapes hard (H2 nameability, stage 0 naming check)
if NAMING:
    run("run_nameability.py", "--model", MODEL, *LIMIT)
else:
    print(f"part {PART}: the naming step runs in the model's other part")

In [ ]:
# H1-H8 with 95% bootstrap intervals, as fixed in docs/study3_methodology.pdf
# the trial also runs it once on its few answers, only to catch errors before the long run
missing = [name for name in ("shapes", "hard", "objects", "lines") if not (PREDS / f"preds_{name}_{MODEL}.csv").exists()]
if not (PREDS / f"nameability_{MODEL}.csv").exists():
    missing.append("naming step")
if missing:
    print(f"analysis skipped, not in this run yet: {missing}. It runs once all parts' preds files are in one folder")
else:
    code = run("analysis.py", "--model", MODEL, must_pass=STAGE == "full")
    if code:
        print(f"analysis exit code {code} on the trial answers: check the error above before the full run")

In [ ]:
# checks of this run, and for the trial an estimate of the full run time
text = "".join(OUTPUT)
failed = [m.group(1) for m in re.finditer(r"^(\w+): scorer exit code (?!0\b)(\d+)", text, re.M)]
print("scorers that failed:", failed or "none")
for file in sorted(PREDS.glob(f"preds_*_{MODEL}.csv")):
    rows = file.read_text(encoding="utf-8").splitlines()
    print(f"\n{file.name}: {len(rows) - 1} answers, first ones:")
    print("\n".join(rows[1:4]))

if STAGE == "trial":
    from datasets import concatenate_datasets, load_dataset

    REPOS = {"shapes": "atml-shapes-vqa/shapes-vqa-distractors", "hard": "atml-shapes-vqa/shapes-vqa-hard",
             "objects": "atml-shapes-vqa/objects-vqa", "lines": "atml-shapes-vqa/lines-vqa"}
    NAMED = ["squiggles_2", "squiggles_4", "squiggles_8", "squiggles_16", "other_shapes"]
    per_image = {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+): \d+ images in [\d.]+ min \(([\d.]+) s per image\)", text, re.M)}
    per_shape = float(re.search(r"^nameability: \d+ shapes in [\d.]+ min \(([\d.]+) s per shape\)", text, re.M).group(1))
    hours = {}
    for name, repo in REPOS.items():
        data = load_dataset(repo)  # already in the cache from the trial
        data = concatenate_datasets([data[split] for split in data])
        hours[name] = len(data) * per_image[name] / 3600
        print(f"{name}: {len(data)} images x {per_image[name]:.2f} s = {hours[name]:.1f} h")
        if name == "hard":
            meta = data.select_columns(["version", "objects"])
            shapes = 0
            for version, objects in zip(meta["version"], meta["objects"]):
                if version in NAMED:
                    objects = json.loads(objects) if isinstance(objects, str) else objects
                    shapes += sum(o["role"] == "distractor" for o in objects)
            hours["nameability"] = shapes * per_shape / 3600
            print(f"nameability: {shapes} shapes x {per_shape:.2f} s = {hours['nameability']:.1f} h")
    # about 10 min per script start (model loading) on top of the measured time
    part1 = hours["shapes"] + hours["hard"] + 0.2
    part2 = hours["objects"] + hours["lines"] + hours["nameability"] + 0.5
    total = part1 + part2 - 0.2
    print(f"\nestimated full run: {total:.1f} h in one version, or part 1 {part1:.1f} h + part 2 {part2:.1f} h")
    print("(slightly on the high side: the trial's time also includes reading the images it skipped)")
    if total <= 10.5:
        print("-> fits in Kaggle's 12 h limit: run STAGE='full', PART='all'")
    elif max(part1, part2) <= 10.5:
        print("-> too long for one version: run STAGE='full' with PART='1', then PART='2'")
    else:
        print("-> even one part may pass 12 h: tell the study owner before the full run")